# 02 - OOD Model Generalization Evaluation & Benchmark

This notebook executes the out-of-distribution (OOD) generalization testing campaign for the WiFi CSI presence detection system.
The objective is to evaluate pre-trained models on unseen physical environments and inter-node distances (other than the baseline 2.0-meter configuration).

### Validation Principles:
- **Zero Retraining Policy**: Models are evaluated strictly as pre-trained inference artifacts.
- **Full-Feature vs Lightweight Comparison**: Compares the 4 full-feature models (648 features) against their 4 corresponding lightweight variants (20 features across 5 optimal subcarriers).
- **Subcarrier Alignment**: Feature extraction strictly uses the reference 162-subcarrier mapping (`data/03_processed/valid_subcarrier_mapping.csv`).
- **Publication-Ready Figures & Reports**: All metrics and figures are exported to `outputs/generalization/`.


## 1. Environment Setup & Package Imports

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

# Locate repository root
_p = Path.cwd()
for PROJECT_ROOT in [_p, *_p.parents]:
    if (PROJECT_ROOT / "requirements.txt").exists():
        break

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Established wifi_csi package imports
from wifi_csi.parsing.metadata_parser import discover_sessions, load_metadata
from wifi_csi.signal.amplitudes import load_session_arrays
from wifi_csi.features.extractor import build_feature_dataset
from wifi_csi.evaluation.metrics import compute_metrics, false_alarm_rate
from wifi_csi.evaluation.plotting import plot_confusion_matrix, plot_model_comparison

# Target directories
DATA_DIR = PROJECT_ROOT / "data" / "01_raw" / "generalization"
MODELS_DIR = PROJECT_ROOT / "models" / "registry"
OUTPUTS_DIR = PROJECT_ROOT / "outputs" / "generalization"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
MAPPING_FILE = PROJECT_ROOT / "data" / "03_processed" / "valid_subcarrier_mapping.csv"

# Plotting configuration
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3
})

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Data source: {DATA_DIR.resolve()}")
print(f"Outputs target: {OUTPUTS_DIR.resolve()}")
print(f"Model registry: {MODELS_DIR.resolve()}")


## 2. Session Discovery and Signal Loading

Scan `data/01_raw/generalization/` for valid sessions, parse metadata, and load raw CSI arrays.

In [ ]:
discovered_df = discover_sessions(DATA_DIR, required_status="VALID")
print(f"Discovered {len(discovered_df)} valid session(s) in {DATA_DIR.name}:")
display(discovered_df[["session_id", "label_name", "metadata_status", "file_size_mb", "csv_filename"]])

loaded_sessions = []
session_metadata_records = []

for rec in discovered_df.to_dict("records"):
    session = load_session_arrays(rec["csv_path"], rec["metadata_path"])
    loaded_sessions.append(session)
    
    meta = session["metadata"]
    env_info = meta.get("environment", {})
    setup_info = meta.get("setup", {})
    nodes_info = setup_info.get("nodes", {})
    
    session_metadata_records.append({
        "session_id": session["session_id"],
        "label_name": session["label_name"],
        "label": session["label"],
        "environment_id": env_info.get("environment_id", "unknown"),
        "tx_rx_los_distance_m": nodes_info.get("tx_rx_los_distance_m", env_info.get("tx_rx_distance_m")),
        "valid_rows": session["metrics"]["valid_rows"],
        "effective_rate_hz": round(session["effective_rate_hz"], 2),
        "duration_s": round(session["metrics"]["duration_s"], 1),
        "active_interval": f"{session['active_start']} -> {session['active_end']}",
        "scenario_notes": env_info.get("scenario_notes", "")
    })

session_summary_df = pd.DataFrame(session_metadata_records)
display(session_summary_df)


## 3. Subcarrier Alignment & Feature Extraction Pipeline

Extract features using the reference 162-subcarrier mapping (`valid_subcarrier_mapping.csv`).
This guarantees that `sc000` through `sc161` correspond identically to the exact physical subcarrier frequencies that the pre-trained pipelines were trained on.

In [ ]:
# Load the 162-subcarrier mapping used in training to guarantee exact feature alignment
mapping_df = pd.read_csv(MAPPING_FILE)
raw_indices = mapping_df["raw_subcarrier_index"].values
n_raw_subcarriers = loaded_sessions[0]["amplitude_matrix"].shape[1]

reference_mask = np.zeros(n_raw_subcarriers, dtype=bool)
reference_mask[raw_indices] = True
print(f"Reference mask initialized: {int(reference_mask.sum())} subcarriers matching training registry.")

# Build standard feature dataset (2.0s non-overlapping windows)
features_df, out_mapping_df, processing_details = build_feature_dataset(
    loaded_sessions,
    shared_mask=reference_mask,
    window_seconds=2.0
)

# Merge session-level metadata into windowed dataset
meta_lookup = session_summary_df.set_index("session_id")[["environment_id", "tx_rx_los_distance_m", "scenario_notes"]].to_dict("index")
features_df["environment_id"] = features_df["session_id"].map(lambda sid: meta_lookup.get(sid, {}).get("environment_id", "unknown"))
features_df["tx_rx_los_distance_m"] = features_df["session_id"].map(lambda sid: meta_lookup.get(sid, {}).get("tx_rx_los_distance_m", np.nan))
features_df["scenario_notes"] = features_df["session_id"].map(lambda sid: meta_lookup.get(sid, {}).get("scenario_notes", ""))

y_true = features_df["label"].to_numpy(dtype=int)
print(f"\nFeature dataset built successfully:")
print(f"  Total window samples: {len(features_df)}")
print(f"  Total columns: {features_df.shape[1]}")
print(f"  Class balance: {dict(pd.Series(y_true).value_counts())} (0=empty, 1=occupied)")
print(f"  Tested environments: {list(features_df['environment_id'].unique())}")
print(f"  Tested distances (m): {sorted(list(features_df['tx_rx_los_distance_m'].unique()))}")


## 4. Pre-Trained Model Registry Loading & Verification

Load the 8 pre-trained model bundles from `models/registry/`:
- 4 Full Feature Models (648 features across all 162 subcarriers)
- 4 Lightweight Models (20 features across top 5 optimal subcarriers: `sc041, sc040, sc042, sc047, sc012`)

In [ ]:
MODEL_SPECS = [
    # Full feature models (648 features)
    {"name": "gradient_boosting", "bundle": "gradient_boosting_bundle", "family": "Gradient Boosting", "type": "Full", "features": 648},
    {"name": "random_forest", "bundle": "random_forest_bundle", "family": "Random Forest", "type": "Full", "features": 648},
    {"name": "svm", "bundle": "svm_bundle", "family": "SVM", "type": "Full", "features": 648},
    {"name": "mlp", "bundle": "mlp_bundle", "family": "MLP", "type": "Full", "features": 648},
    # Lightweight models (20 features)
    {"name": "gradient_boosting_light", "bundle": "gradient_boosting_light_bundle", "family": "Gradient Boosting", "type": "Lightweight", "features": 20},
    {"name": "random_forest_light", "bundle": "random_forest_light_bundle", "family": "Random Forest", "type": "Lightweight", "features": 20},
    {"name": "svm_light", "bundle": "svm_light_bundle", "family": "SVM", "type": "Lightweight", "features": 20},
    {"name": "mlp_light", "bundle": "mlp_light_bundle", "family": "MLP", "type": "Lightweight", "features": 20},
]

loaded_models = {}
for spec in MODEL_SPECS:
    bundle_path = MODELS_DIR / spec["bundle"]
    pipeline = joblib.load(bundle_path / "pipeline.joblib")
    card_path = bundle_path / "model_card.json"
    card = json.loads(card_path.read_text()) if card_path.exists() else {}
    
    loaded_models[spec["name"]] = {
        "pipeline": pipeline,
        "card": card,
        "spec": spec,
        "expected_features": list(pipeline.feature_names_in_)
    }
    print(f"Loaded: {spec['name']:25s} | Type: {spec['type']:11s} | In-features: {len(pipeline.feature_names_in_)}")


## 5. Model Inference & Comprehensive Metric Evaluation

Run inference on the OOD feature dataset and compute all evaluation metrics:
- **Accuracy**
- **Macro F1-Score**
- **Weighted F1-Score**
- **Sensitivity (TPR / Recall)**
- **Specificity (TNR)**
- **False Alarm Rate (FPR)**
- **Confusion Matrix Counts (TP, TN, FP, FN)**
- **In-Distribution vs Out-of-Distribution F1 Degradation**

In [ ]:
results_list = []
predictions_dict = {}

for name, model_data in loaded_models.items():
    pipe = model_data["pipeline"]
    spec = model_data["spec"]
    card = model_data["card"]
    
    feat_cols = model_data["expected_features"]
    X = features_df[feat_cols]
    
    y_pred = pipe.predict(X)
    predictions_dict[name] = y_pred
    
    acc = float(accuracy_score(y_true, y_pred))
    f1_mac = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    f1_wt = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = tp = 0
    
    sens = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
    spec_val = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
    far = float(fp / (tn + fp)) if (tn + fp) > 0 else 0.0
    
    baseline_f1 = card.get("metadata", {}).get("test_metrics", {}).get("f1_macro", card.get("metadata", {}).get("f1_macro", np.nan))
    delta_f1 = f1_mac - baseline_f1 if not np.isnan(baseline_f1) else np.nan
    
    results_list.append({
        "model_name": name,
        "family": spec["family"],
        "type": spec["type"],
        "n_features": spec["features"],
        "accuracy": acc,
        "f1_macro": f1_mac,
        "f1_weighted": f1_wt,
        "sensitivity": sens,
        "specificity": spec_val,
        "false_alarm_rate": far,
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "baseline_f1_macro": baseline_f1,
        "f1_degradation": delta_f1
    })

metrics_df = pd.DataFrame(results_list)
display(metrics_df[["model_name", "type", "n_features", "accuracy", "f1_macro", "f1_weighted", "sensitivity", "specificity", "false_alarm_rate", "baseline_f1_macro", "f1_degradation"]])


## 6. Stratified Robustness Analysis (Distance & Environment)

Break down model generalization across unseen physical distances and environments.

In [ ]:
# Stratified analysis by inter-node distance
dist_rows = []
for name, y_pred in predictions_dict.items():
    spec = loaded_models[name]["spec"]
    eval_df = features_df[["tx_rx_los_distance_m", "label"]].copy()
    eval_df["y_true"] = y_true
    eval_df["y_pred"] = y_pred
    
    for dist, g in eval_df.groupby("tx_rx_los_distance_m", observed=True):
        dist_acc = float(accuracy_score(g["y_true"], g["y_pred"]))
        dist_f1 = float(f1_score(g["y_true"], g["y_pred"], average="macro", zero_division=0))
        dist_rows.append({
            "model_name": name,
            "family": spec["family"],
            "type": spec["type"],
            "distance_m": dist,
            "n_samples": len(g),
            "accuracy": dist_acc,
            "f1_macro": dist_f1
        })

metrics_by_dist_df = pd.DataFrame(dist_rows)
print("Distance Breakdown (sample):")
display(metrics_by_dist_df.head(8))

# Stratified analysis by environment
env_rows = []
for name, y_pred in predictions_dict.items():
    spec = loaded_models[name]["spec"]
    eval_df = features_df[["environment_id", "label"]].copy()
    eval_df["y_true"] = y_true
    eval_df["y_pred"] = y_pred
    
    for env, g in eval_df.groupby("environment_id", observed=True):
        env_acc = float(accuracy_score(g["y_true"], g["y_pred"]))
        env_f1 = float(f1_score(g["y_true"], g["y_pred"], average="macro", zero_division=0))
        env_rows.append({
            "model_name": name,
            "family": spec["family"],
            "type": spec["type"],
            "environment_id": env,
            "n_samples": len(g),
            "accuracy": env_acc,
            "f1_macro": env_f1
        })

metrics_by_env_df = pd.DataFrame(env_rows)
print("\nEnvironment Breakdown (sample):")
display(metrics_by_env_df.head(8))


## 7. Comparative Visualizations (Full-Feature vs Lightweight Models)

Generate comparative visual figures and confusion matrix grids.

In [ ]:
# Figure 1: Model Comparison (Full vs Light across Accuracy, Macro F1, Sensitivity, Specificity)
fig, ax = plt.subplots(figsize=(12, 6))
plot_data = metrics_df.copy()
plot_data["display_name"] = plot_data["family"] + " (" + plot_data["type"] + ")"

metric_cols = ["accuracy", "f1_macro", "f1_weighted", "sensitivity", "specificity"]
x = np.arange(len(plot_data))
width = 0.16

colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd", "#8c564b"]
for i, (col, color) in enumerate(zip(metric_cols, colors)):
    offset = (i - len(metric_cols) / 2 + 0.5) * width
    ax.bar(x + offset, plot_data[col], width, label=col.replace("_", " ").title(), color=color, alpha=0.9)

ax.set_ylabel("Metric Score (0.0 - 1.0)")
ax.set_title("OOD Generalization Performance: Full-Feature vs Lightweight Models", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(plot_data["display_name"], rotation=30, ha="right")
ax.set_ylim(0, 1.15)
ax.axhline(0.90, color="red", linestyle="--", alpha=0.6, label="F1 Target Threshold (0.90)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

fig1_path = OUTPUTS_DIR / "model_comparison_full_vs_light.png"
plt.savefig(fig1_path, dpi=200, bbox_inches="tight")
print(f"Saved Figure 1: {fig1_path}")
plt.show()


In [ ]:
# Figure 2: Confusion Matrices Grid (2 Rows x 4 Columns)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
families = ["Gradient Boosting", "Random Forest", "SVM", "MLP"]

for col_idx, fam in enumerate(families):
    # Row 0: Full model
    full_name = metrics_df[(metrics_df["family"] == fam) & (metrics_df["type"] == "Full")]["model_name"].iloc[0]
    y_pred_full = predictions_dict[full_name]
    plot_confusion_matrix(y_true, y_pred_full, ax=axes[0, col_idx], title=f"{fam} (Full, 648 feat)")
    
    # Row 1: Light model
    light_name = metrics_df[(metrics_df["family"] == fam) & (metrics_df["type"] == "Lightweight")]["model_name"].iloc[0]
    y_pred_light = predictions_dict[light_name]
    plot_confusion_matrix(y_true, y_pred_light, ax=axes[1, col_idx], title=f"{fam} (Light, 20 feat)")

plt.suptitle("OOD Generalization Confusion Matrices: Full-Feature vs Lightweight", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()

fig2_path = OUTPUTS_DIR / "confusion_matrices_all_models.png"
plt.savefig(fig2_path, dpi=200, bbox_inches="tight")
print(f"Saved Figure 2: {fig2_path}")
plt.show()


In [ ]:
# Figure 3: Robustness across Inter-Node Distances
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(
    data=metrics_by_dist_df,
    x="distance_m",
    y="f1_macro",
    hue="family",
    style="type",
    markers=True,
    dashes=False,
    ax=ax,
    linewidth=2.2,
    markersize=9
)
ax.set_title("Macro F1-Score vs Inter-Node Physical Distance", fontweight="bold")
ax.set_xlabel("Inter-Node Distance (m)")
ax.set_ylabel("Macro F1-Score")
ax.set_ylim(-0.05, 1.05)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

fig3_path = OUTPUTS_DIR / "robustness_by_distance.png"
plt.savefig(fig3_path, dpi=200, bbox_inches="tight")
print(f"Saved Figure 3: {fig3_path}")
plt.show()


In [ ]:
# Figure 4: Robustness across Physical Environments
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=metrics_by_env_df,
    x="environment_id",
    y="f1_macro",
    hue="family",
    ax=ax,
    palette="tab10"
)
ax.set_title("Macro F1-Score across Physical Environments", fontweight="bold")
ax.set_xlabel("Environment Identifier")
ax.set_ylabel("Macro F1-Score")
ax.set_ylim(0, 1.1)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

fig4_path = OUTPUTS_DIR / "robustness_by_environment.png"
plt.savefig(fig4_path, dpi=200, bbox_inches="tight")
print(f"Saved Figure 4: {fig4_path}")
plt.show()


## 8. Artifact Export & Summary Report Generation

Save all metrics, tables, and serialized reports to `outputs/generalization/`.

In [ ]:
# 1. Save summary CSVs
summary_csv = OUTPUTS_DIR / "metrics_summary.csv"
metrics_df.to_csv(summary_csv, index=False)

dist_csv = OUTPUTS_DIR / "metrics_by_distance.csv"
metrics_by_dist_df.to_csv(dist_csv, index=False)

env_csv = OUTPUTS_DIR / "metrics_by_environment.csv"
metrics_by_env_df.to_csv(env_csv, index=False)

# 2. Save summary JSONs
summary_json = OUTPUTS_DIR / "metrics_summary.json"
metrics_df.to_json(summary_json, orient="records", indent=2)

# 3. Save comprehensive generalization report
report_data = {
    "campaign": "OOD Generalization Testing Appendix",
    "total_sessions": len(loaded_sessions),
    "total_windows": len(features_df),
    "environments": list(features_df["environment_id"].unique()),
    "distances_m": [float(d) for d in sorted(features_df["tx_rx_los_distance_m"].unique())],
    "models_evaluated": len(metrics_df),
    "metrics": metrics_df.to_dict(orient="records"),
    "generated_figures": [
        "model_comparison_full_vs_light.png",
        "confusion_matrices_all_models.png",
        "robustness_by_distance.png",
        "robustness_by_environment.png"
    ]
}

report_json = OUTPUTS_DIR / "generalization_report.json"
with open(report_json, "w", encoding="utf-8") as f:
    json.dump(report_data, f, indent=2)

print("=" * 65)
print("GENERALIZATION BENCHMARK COMPLETE")
print(f"Artifacts exported to: {OUTPUTS_DIR.resolve()}")
print(f"  - {summary_csv.name}")
print(f"  - {summary_json.name}")
print(f"  - {dist_csv.name}")
print(f"  - {env_csv.name}")
print(f"  - {report_json.name}")
print("=" * 65)
